[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/mobillity-univ/module-6-telling-the-story/notebook-6.5-from-colab-to-stakeholder-deck.ipynb)


# From a verified Colab analysis to a deck the transport authority can read

**The question.** We have already measured how often FGC trains run in the evening — one number per line, for the 20:00–24:00 window on a representative weekday. FGC is Ferrocarrils de la Generalitat de Catalunya, the operator behind Barcelona's metro, commuter-rail and funicular lines (there is no bus in this network). Next week that finding has to be presented to the transport authority. How do we turn an analysis that lives in code and charts into a short, professional slide deck — without rebuilding it slide by slide by hand?

**Why it's worth asking.** An analysis is only useful once the people who fund and plan the service can see it. The slow way is to open PowerPoint and retype every number by hand; the fast, reliable way is to keep the verified analysis as the source and have an assistant write the code that assembles the deck for us.

**The data.** Each row of the timetable is a scheduled departure — a trip leaving a stop at a given time. From those rows we build one row per line: its evening departures, its departures per hour, and the longest a rider waits between trains in the evening. One honest caveat: these figures describe a single representative weekday and the evening window only, and they were checked against the real feed earlier — here we reuse that verified result, we don't re-audit it.

**The method — five moves.** Structure the message as a pyramid (the answer first, then the reasons), draft a brief for this audience, have the assistant write the code that builds the deck, review what it built, and refine it in a round or two.

In [1]:
# --- Setup: install the library this notebook uses -------------------------
# On Google Colab the first run installs python-pptx; run locally it is a no-op
# when the package is already present. Safe to re-run.
import importlib.util, subprocess, sys

if importlib.util.find_spec("pptx") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"], check=True)
print("SETUP_OK: python-pptx available")

SETUP_OK: python-pptx available


## 1. Bring the verified finding in as a table

Before any slides exist, we want the finding itself in front of us as data - not a number we half-remember. The whole point of automating the deck is that every figure on a slide comes straight from this table, so nothing is retyped or rounded by hand. The finding was computed and audited earlier (story-5.4); here we reuse that verified result. It is a short table - one row per line - ranging from a few minutes' wait to a couple of hours.

In [2]:
"""The verified per-line evening-service table - carried forward, not recomputed here.

One row per FGC line for a representative weekday evening (20:00-24:00), validated against
the real feed earlier in the course (story-5.4). Columns: line code, mode, evening
departures, departures/hour, worst-direction wait (min)."""
import pandas as pd

line_table = pd.DataFrame(
    [
        ("FV", "funicular", 80, 20.0, 6),
        ("L7", "metro", 56, 14.0, 9),
        ("L12", "metro", 45, 11.2, 11),
        ("L6", "metro", 39, 9.8, 13),
        ("S1", "rail", 25, 6.2, 24),
        ("S2", "rail", 25, 6.2, 24),
        ("L8", "metro", 16, 4.0, 34),
        ("R63", "rail", 3, 0.8, 80),
        ("S8", "rail", 14, 3.5, 80),
        ("R53", "rail", 2, 0.5, 120),
        ("S3", "rail", 5, 1.2, 120),
        ("S4", "rail", 4, 1.0, 120),
        ("R5", "rail", 7, 1.8, 240),
        ("R6", "rail", 6, 1.5, 240),
        ("RL1", "rail", 3, 0.8, 240),
    ],
    columns=["route_short_name", "mode", "evening_departures", "dep_per_hour", "evening_wait_min"],
)

print(line_table.to_string(index=False))
print("LINES:", line_table.route_short_name.nunique(),
      "| EVENING_TRIPS:", int(line_table.evening_departures.sum()))

route_short_name      mode  evening_departures  dep_per_hour  evening_wait_min
              FV funicular                  80          20.0                 6
              L7     metro                  56          14.0                 9
             L12     metro                  45          11.2                11
              L6     metro                  39           9.8                13
              S1      rail                  25           6.2                24
              S2      rail                  25           6.2                24
              L8     metro                  16           4.0                34
             R63      rail                   3           0.8                80
              S8      rail                  14           3.5                80
             R53      rail                   2           0.5               120
              S3      rail                   5           1.2               120
              S4      rail                   4      

Fifteen lines run in the evening, 330 departures in all. The spread is stark: the funicular, FV, comes every six minutes, while the R5, R6 and RL1 lines leave riders waiting up to 240 minutes — four hours — between trains. That unevenness is the story the deck has to tell, and it is now sitting in a table every slide can point at.

## 2. Structure the message as a pyramid

A table is not a message. Stakeholders in a room want the conclusion first, then the few reasons behind it — the shape Barbara Minto called a pyramid: one main point on top, supported by a small number of reasons, each backed by evidence. Drafting that from memory is where numbers quietly get rounded or overstated, so instead we hand the assistant the table we just built and ask it to structure the finding for us — one headline, three reasons, each tied to a figure in the table. The alternative — walking the audience through the table row by row and hoping they reach the conclusion themselves — is exactly what loses a room. A prompt for that:

```markdown
You're helping me turn a transit-schedule finding into a short presentation for a
regional transport authority — decision-makers, not analysts. Here's the table I
just computed: one row per line for a weekday evening (20:00-24:00), showing each
line's evening departures, its departures per hour, and the longest a rider waits
between trains in the worst direction, sorted shortest wait to longest.

line  mode        evening_departures  dep_per_hour  wait_min
FV    funicular   80                  20.0          6
L7    metro       56                  14.0          9
L12   metro       45                  11.2          11
L6    metro       39                  9.8           13
S1    rail        25                  6.2           24
S2    rail        25                  6.2           24
L8    metro       16                  4.0           34
R63   rail        3                   0.8           80
S8    rail        14                  3.5           80
R53   rail        2                   0.5           120
S3    rail        5                   1.2           120
S4    rail        4                   1.0           120
R5    rail        7                   1.8           240
R6    rail        6                   1.5           240
RL1   rail        3                   0.8           240

Structure this as a Minto pyramid I can build a deck around: one plain-language
headline stating the single main finding, then exactly three supporting reasons
under it. Ground each reason in a real number from the table — (1) the best line's
wait versus the worst, and how many times longer that is; (2) how many lines run
less than one train every two hours after 20:00, and which ones; (3) the gap in
departures per hour between the best and worst line. Keep every line plain enough
for a non-technical reader: no schedule jargon, no code — just the message.
```

Its pyramid comes back like this:

```markdown
**Headline.** Evening service on FGC is uneven — riders on the thinner-served lines wait far longer for a train than riders on the core network.

**Reason 1 — the gap is large.** The best-served line runs a train every 6 minutes in the evening; the worst waits 240 minutes — four hours — between trains. That is 40 times longer.

**Reason 2 — several lines are barely served.** Six lines run less than one train every two hours after 20:00: R53, S3, S4, R5, R6, and RL1.

**Reason 3 — the imbalance is structural.** The best line manages 20 departures an hour; the worst manages half a departure an hour — one train every two hours.
```

Before trusting any of it, we check the pyramid back against the table — and it holds. The best-to-worst gap is real: six minutes against 240, forty times longer. Six lines really do run under one train every two hours. And the departures-per-hour difference — twenty on the best line against half on the worst — is right there in the data. The headline and its three reasons are the spine the slides will now follow.

## 3. Draft the brief, one line per slide

Now we shape the message for this particular audience. The transport authority decides where to add service, so the deck has to carry them from the finding to a recommendation they can act on — not drown them in method. Before building anything, we ask the assistant to turn the pyramid into a brief: one line per intended slide, in the order we would present them. Settling the brief first is what keeps the deck aimed at the room it is for — it is far cheaper to move a line here than to rebuild a slide later — and every line has to trace back to the headline or one of the reasons, with no new numbers introduced. A prompt for that:

```markdown
Now turn that pyramid into a slide-by-slide brief for the same transport-authority
audience — one line per slide, in the order I'd present them. Keep it to seven
slides: a title slide, an executive-summary slide carrying the headline, an
evidence slide for the frequency chart, a best-versus-worst slide, a slide naming
the worst-served lines, a "what it means" slide, and a recommendations slide.
Every line must come from the headline or one of the three reasons — introduce no
new numbers — and the recommendation should be something the authority can act on.
Give it back to me as "Slide N — Heading: one line of content."
```

The brief it proposes:

```markdown
**Slide 1 — FGC Evening Service Frequency:** Weekday evening, 20:00-24:00 — prepared for the transport authority.

**Slide 2 — Executive summary:** Evening service on FGC is uneven — riders on the thinner-served lines wait far longer for a train than riders on the core network.

**Slide 3 — Evidence:** Minutes between evening trains, by line.

**Slide 4 — Best vs worst:** The best-served line runs every 6 minutes; the worst waits 240 minutes — 40 times longer.

**Slide 5 — Who is affected:** Six lines run less than one train every two hours after 20:00: R53, S3, S4, R5, R6, and RL1.

**Slide 6 — What it means:** Riders on the outer rail lines face far longer evening waits than the core network.

**Slide 7 — Recommendations:** Add evening trips on the lines waiting two hours or more, then re-check after the next timetable change.
```

Seven slides, and every line on them traces back to something we already measured — the executive summary is the headline, "best vs worst" and "who is affected" are two of the reasons, and the recommendation follows from them. This short list is the spec the deck gets built from.

## 4. Build the deck — and read the first error

With the brief settled, the last decision is the format. python-pptx builds a PowerPoint file you can open, forward and present in a meeting; reveal.js builds an interactive web page that lives at a link. The authority wants a file to open and pass around, so we take the PowerPoint route. This is exactly the point where it is tempting to open PowerPoint and assemble the slides by hand — and that is the move to skip. The brief and the numbers are already here, so instead we have the assistant write the code that turns the brief into slides, and we run it.

```markdown
Using python-pptx, write Python that builds a PowerPoint deck from this brief - one
slide per line, each with its heading as a bold title and the content below it - puts
the evening-frequency bar chart on the evidence slide, and saves the file.
```

In [3]:
"""
Build the PowerPoint deck from the agreed brief - first run of the assistant's code.
Data: the brief we just settled, written down as data the code can loop over.
"""
# 0. Settings you can change
DECK = "story-6.5-fgc-evening-service.pptx"

# 1. The agreed brief, as (heading, one-line content) per slide
brief = [
    ("FGC Evening Service Frequency", "Weekday evening, 20:00-24:00 — prepared for the transport authority."),
    ("Executive summary", "Evening service on FGC is uneven — riders on the thinner-served lines wait far longer for a train than riders on the core network."),
    ("Evidence", "Minutes between evening trains, by line."),
    ("Best vs worst", "The best-served line runs every 6 minutes; the worst waits 240 minutes — 40 times longer."),
    ("Who is affected", "Six lines run less than one train every two hours after 20:00: R53, S3, S4, R5, R6, and RL1."),
    ("What it means", "Riders on the outer rail lines face far longer evening waits than the core network."),
    ("Recommendations", "Add evening trips on the lines waiting two hours or more, then re-check after the next timetable change."),
]

# 2. Start an empty presentation
from pptx import Presentation
prs = Presentation()
blank = prs.slide_layouts[6]

# 3. Put each brief line on its own slide: a bold title and the content below it
for heading, line in brief:
    slide = prs.slides.add_slide(blank)
    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(9), Inches(1)).text_frame
    title.text = heading
    title.paragraphs[0].runs[0].font.size = Pt(32)
    title.paragraphs[0].runs[0].font.bold = True
    body = slide.shapes.add_textbox(Inches(0.5), Inches(1.6), Inches(9), Inches(4)).text_frame
    body.word_wrap = True
    body.text = line

prs.save(DECK)
print("DECK_SAVED:", DECK)

NameError: name 'Inches' is not defined

It doesn't run. The error is short: the name `Inches` is not defined. Read literally, that is all it is — the code positions text boxes in inches and sets font sizes in points, but it never imported the two helpers that provide those units. This is a missing import, not a wrong approach: the slides were laid out correctly, the code just reached for a tool it hadn't picked up. The fix is one line, not a rewrite.

## 5. Fix the import and build it for real

The error told us exactly what to add — the size helpers python-pptx uses for positions and font sizes. We ask the assistant how to solve it, it brings in that one import that is missing, and we run the build again. Nothing else about the approach changes. This is the ordinary rhythm of working with generated code: run it, read what it says, ask for help to the assistant, run it again.

In [4]:
"""
Fix the missing import and build the deck for real, with the evidence chart.
Data: brief (slide list) + line_table (for the chart).
"""
# 0. Settings you can change
DECK = "story-6.5-fgc-evening-service.pptx"
CHART = "story-6.5-evening-frequency.png"

# 1. The one-line fix: import the size helpers the code uses
from pptx import Presentation
from pptx.util import Inches, Pt

# 2. Render the evidence chart the deck will embed
import matplotlib
import matplotlib.pyplot as plt
ranked = line_table.sort_values("evening_wait_min")
bar_colors = ["#10B981" if w <= 15 else "#F59E0B" if w < 120 else "#DC2626" for w in ranked.evening_wait_min]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranked.route_short_name, ranked.evening_wait_min, color=bar_colors)
ax.invert_yaxis()
ax.set_xlabel("Minutes between trains - worst direction, evening 20:00-24:00")
ax.set_title("FGC evening service is uneven by line")
for i, w in enumerate(ranked.evening_wait_min):
    ax.text(w + 3, i, f"{int(w)}m", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(CHART, dpi=120)
plt.close(fig)

# 3. Build the slides, dropping the chart onto the Evidence slide
prs = Presentation()
blank = prs.slide_layouts[6]
for heading, line in brief:
    slide = prs.slides.add_slide(blank)
    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.4), Inches(9), Inches(1)).text_frame
    title.text = heading
    title.paragraphs[0].runs[0].font.size = Pt(32)
    title.paragraphs[0].runs[0].font.bold = True
    body = slide.shapes.add_textbox(Inches(0.5), Inches(1.6), Inches(9), Inches(4)).text_frame
    body.word_wrap = True
    body.text = line
    if heading == "Evidence":
        slide.shapes.add_picture(CHART, Inches(0.5), Inches(2.4), width=Inches(8))
prs.save(DECK)

print("CHART_SAVED:", CHART)
print("DECK_SAVED:", DECK)
print("SLIDE_COUNT:", len(prs.slides._sldIdLst))

CHART_SAVED: story-6.5-evening-frequency.png
DECK_SAVED: story-6.5-fgc-evening-service.pptx
SLIDE_COUNT: 7


This time it saves. Seven slides, the chart embedded on the evidence slide, every number carried straight from the verified table — nothing retyped. The deck was assembled entirely by code from the brief, which is what lets us trust the figures on it.

## 6. Review what the code built

A saved file is not a finished deck. Generated code can quietly drop a slide, repeat one, or skip the chart, and it is far cheaper to catch that here than in front of the authority. So before anyone sees it, we open the deck back up and check it against the brief: are all the slides there, in order; is the chart actually embedded; does the headline sit on the summary slide. The check takes a few seconds, and it is the habit that keeps a generated deck honest.

## 7. Refine it in one short round

The first deck is a strong draft, not the finished article — that is normal for anything an assistant produces, slides included. So we look it over the way the audience will and send back small, specific fixes. For example, you might find the recommendations read too small from the back of a room and ask for that slide's text enlarged, or want the chart reordered, or a line tightened. Each is one short instruction and a re-run, not a rebuild. Here we make just that one change and save a fresh copy.

```markdown
For example: the recommendations slide is hard to read from the back of the room -
make its body text 24 point and save the deck as a new version.
```

In [5]:
"""
One refinement round: make the recommendations readable from the back of the room.
Data: the saved deck; we re-save a new version rather than rebuild.
"""
from pptx import Presentation
from pptx.util import Pt

# 0. Settings you can change
REVISED = "story-6.5-fgc-evening-service-v2.pptx"
BIGGER_PT = 24

# 1. Find the recommendations slide and enlarge only its body text
prs = Presentation(DECK)
for slide in prs.slides:
    shapes = list(slide.shapes)
    if shapes and shapes[0].has_text_frame and shapes[0].text_frame.text == "Recommendations":
        body = shapes[1].text_frame          # the second box on the slide is the body
        for run in body.paragraphs[0].runs:
            run.font.size = Pt(BIGGER_PT)
prs.save(REVISED)

print("REVISED_SAVED:", REVISED)
print("RECOMMENDATIONS_PT:", BIGGER_PT)

REVISED_SAVED: story-6.5-fgc-evening-service-v2.pptx
RECOMMENDATIONS_PT: 24


The revised deck saves with the recommendations text enlarged. In practice it is two or three rounds like this — a small instruction, a re-run — and then the deck is ready to present. At no point did we rebuild it from scratch.

## 8. Read the answer

We started with a finding that lived in code and charts — FGC's evening service is uneven, with the outer lines waiting up to four hours between trains — and we needed it in front of the transport authority next week. Running the five moves, we got there without rebuilding anything by hand: we structured the finding as a pyramid, wrote a brief for the audience, had the assistant write the code that built a seven-slide deck with the evidence chart, reviewed it, and refined it in a round. The analysis never left its source, and every number on every slide came straight from the verified table.

The honest part is the rhythm, not magic. The first code errored and we fixed it in one line; the first deck was a draft we refined. But that whole path — from a screen full of code to a deliverable someone can act on — is one you can repeat for any analysis that has to leave your screen and reach the people who decide what happens next.